# RAG-сервис генерации инструкций

Прототип сервиса, который по запросу пользователя ищет релевантные фрагменты
в базе знаний (портал госуслуг Санкт-Петербурга) и генерирует структурированную
инструкцию с помощью LLM.

# Пайплайн


In [1]:
import sys
from pathlib import Path

# Ноутбук лежит в notebook/, добавляем src/ в sys.path,
# чтобы импортировать модули пайплайна
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from logging_config import setup_logging
from retrieval import retrieve
from generation import generate
from schemas import Instruction

setup_logging()

In [2]:
def print_instruction(instr: Instruction) -> None:
    print("-" * 70)
    print(f"{instr.title}")
    print("-" * 70)
    print(f"Кому адресована: {instr.audience}")
    print(f"Описание: {instr.summary}")
    print()

    print("Необходимые документы:")
    if instr.required_documents:
        for doc in instr.required_documents:
            print(f"   - {doc}")
    else:
        print("   (не указаны)")
    print()

    print("Порядок действий:")
    for i, step in enumerate(instr.steps, start=1):
        print(f"   {i}. {step}")
    print()

    print(f"Сроки: {instr.deadline or 'не указаны'}")
    print(f"Куда обращаться: {instr.where_to_apply or 'не указано'}")
    print(f"Источников: {len(instr.sources)}")

# Пример 1: подача заявления

In [3]:
query = "Как подать заявление в первый класс?"

chunks = retrieve(query, top_k=3)
print(f"Найдено чанков: {len(chunks)}")
for rc in chunks:
    print(f"  [{rc.score:.3f}] {rc.chunk.question}")

instruction = generate(query, chunks)
print()
print_instruction(instruction)

02:54:01 [INFO] retrieval: Загружено чанков: 26
02:54:01 [WARNING] huggingface_hub.utils._http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

02:54:11 [INFO] embeddings: Модель эмбеддингов загружена: intfloat/multilingual-e5-base


Найдено чанков: 3
  [0.897] Как подать заявление?
  [0.870] Как отозвать заявление?
  [0.869] Можно ли подавать заявление, если ребёнок младше 6,5 или старше 8 лет?


02:54:30 [INFO] generation: Валидация прошла успешно



----------------------------------------------------------------------
Подача заявления на зачисление в первый класс
----------------------------------------------------------------------
Кому адресована: Родители
Описание: Инструкция по подаче заявления на зачисление ребёнка в первый класс в Санкт-Петербурге.

Необходимые документы:
   (не указаны)

Порядок действий:
   1. Зайдите на страницу услуги «Зачисление в 1 класс следующего учебного года» на портале gu.spb.ru.
   2. Выберите школу, в которую хотите подать заявление.
   3. Нажмите на кнопку «Подать заявление».
   4. Заполните все необходимые поля формы заявления.
   5. Проверьте правильность введенной информации и нажмите «Отправить».

Сроки: 1 апреля в 00:00
Куда обращаться: Любому удобному многофункциональному центру (МФЦ) или на портале gu.spb.ru
Источников: 1


# Пример 2: путевка в лагерь

In [4]:
query = "Как получить путевку в лагерь?"

chunks = retrieve(query, top_k=3)
print(f"Найдено чанков: {len(chunks)}")
for rc in chunks:
    print(f"  [{rc.score:.3f}] {rc.chunk.question}")

instruction = generate(query, chunks)
print()
print_instruction(instruction)

Найдено чанков: 3
  [0.844] Отдых детей в загородных лагерях
  [0.841] Отдых детей в лагерях дневного пребывания
  [0.812] Когда подавать заявление?


02:55:05 [INFO] generation: Валидация прошла успешно



----------------------------------------------------------------------
Как получить путевку в лагерь для детей
----------------------------------------------------------------------
Кому адресована: Родители детей, проживающих в Санкт-Петербурге
Описание: Инструкция по получению путевки в загородный лагерь для детей, проживающих в Санкт-Петербурге

Необходимые документы:
   (не указаны)

Порядок действий:
   1. Подготовить документы и подать заявление онлайн, в МФЦ или в Санкт Петербургское государственное бюджетное учреждение «Центр оздоровления и отдыха «Молодёжный» (СПб ГБУ ЦОО «Молодёжный») (выбрать удобный способ подачи — для детей работающих родителей; для детей-льготников).
   2. Получить сертификат на частичную или полную оплату стоимости путёвки. В электронном виде сертификат не предоставляется, так как подлежит строгой финансовой отчётности.
   3. Выбрать лагерь из перечня.
   4. Заключить договор с лагерем.
   5. Для всех, кроме льготников: частично оплатить стоимость путёв

# Пример 3: запрос вне базы

In [5]:
query = "Что такое оффсайд в футболе?"

chunks = retrieve(query, top_k=3)
print(f"Найдено чанков: {len(chunks)}")
for rc in chunks:
    print(f"  [{rc.score:.3f}] {rc.chunk.question}")

instruction = generate(query, chunks)
print()
print_instruction(instruction)

02:55:05 [WARNING] retrieval: Ни один чанк не прошел порог 0.78 для запроса: Что такое оффсайд в футболе?


Найдено чанков: 0


ValueError: Не удалось найти информацию в базе данных для вашего запроса. Попробуйте переформулировать.

# Попробовать свой запрос

In [6]:
query = input("Введите ваш запрос: ")

chunks = retrieve(query, top_k=5)
print(f"\nНайдено чанков: {len(chunks)}")
for rc in chunks:
    print(f"  [{rc.score:.3f}] {rc.chunk.question}")

if not chunks:
    print("\nВ базе знаний нет информации по этому запросу.")
else:
    instruction = generate(query, chunks)
    print()
    print_instruction(instruction)


Найдено чанков: 5
  [0.836] Отдых детей в загородных лагерях
  [0.836] Отдых детей в лагерях дневного пребывания
  [0.798] Когда подавать заявление?
  [0.790] Как подтвердить, что ребёнок живёт в том же районе, где находится школа?
  [0.790] Примеры различных ситуаций


02:58:58 [INFO] generation: Валидация прошла успешно



----------------------------------------------------------------------
Как оплатить путевку в детский лагерь
----------------------------------------------------------------------
Кому адресована: Родители детей
Описание: Инструкция по оплате путёвки в загородный лагерь для детей, проживающих в Санкт-Петербурге.

Необходимые документы:
   (не указаны)

Порядок действий:
   1. Подготовить документы и подать заявление онлайн, в МФЦ или в Санкт Петербургское государственное бюджетное учреждение «Центр оздоровления и отдыха «Молодёжный» (выбрать удобный способ подачи — для детей работающих родителей; для детей-льготников).
   2. Получить сертификат на частичную или полную оплату стоимости путёвки.
   3. Выбрать лагерь из перечня.
   4. Заключить договор с лагерем.
   5. Для всех, кроме льготников: частично оплатить стоимость путёвки с учётом сертификата (в случаях, когда город берёт на себя 100% стоимости, оплата не нужна вовсе).
   6. Отправить ребёнка отдыхать. Родитель должен представи